# Data exploration

This notebook shows the profiling of `item.csv` and `event.csv` that supports the design of
`merkle_de_case_study.ipynb`. It is limited to the checks behind the implementation and the data-quality notes.
Section 5 contains post-run checks of the curated tables and the `top_item` datamart; they are kept here so that the
main notebook contains only the pipeline.

**Prerequisite:** run `merkle_de_case_study.ipynb` first. Sections 1–4 read its **layer 1** tables, which are
unchanged string copies of the source files.

In [1]:
from pyspark.sql import functions as F

CATALOG = "merkle_de"

item = spark.table(f"{CATALOG}.layer1_raw.item_raw")
event = spark.table(f"{CATALOG}.layer1_raw.event_raw")

## 1. Item source

In [2]:
# Row count and sample rows
print(f"item rows: {item.count():,}")
display(item.limit(5))

item rows: 2,198


adjective,category,created_at,id,modifier,name,price
fuzzy,contraption,2014-01-15 21:36:09,2512.0,carrying_case,fuzzy contraption carrying_case,150.0
null,instrument,2013-05-14 05:20:50,482.0,refill,instrument refill,35.2
industrial-strength,module,2014-02-04 19:28:32,2446.0,null,industrial-strength module,300.0
digital,tool,2013-02-25 12:23:18,1312.0,carrying_case,digital tool carrying_case,16.5
miniature,device,2013-08-05 17:20:45,3556.0,cleaner,miniature device cleaner,16.5


In [3]:
# Key and attributes: is id a clean integer key, and which attributes can be empty?
display(
    item.select(
        F.count("*").alias("rows"),
        F.count_distinct("id").alias("distinct_id"),
        F.count_if(~F.col("id").rlike(r"^\d+\.0$")).alias("id_not_like_N.0"),
        F.count_if(F.col("adjective").isNull()).alias("empty_adjective"),
        F.count_if(F.col("modifier").isNull()).alias("empty_modifier"),
    )
)

# Price precision and timestamp format
display(
    item.select(
        F.count_if(F.length(F.regexp_extract("price", r"\.(\d+)$", 1)) > 2).alias(
            "price_more_than_2_decimals"
        ),
        F.count_if(F.col("price").cast("double") == 0).alias("zero_price"),
        F.count_if(
            ~F.col("created_at").rlike(r"^\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}$")
        ).alias("created_at_bad_format"),
        F.min("created_at").alias("min_created_at"),
        F.max("created_at").alias("max_created_at"),
    )
)

rows,distinct_id,id_not_like_N.0,empty_adjective,empty_modifier
2198,2198,0,272,766


price_more_than_2_decimals,zero_price,created_at_bad_format,min_created_at,max_created_at
442,156,0,2013-02-22 14:29:32,2014-02-25 23:57:08


**Findings – item**

* **Key:** `id` is unique and always a whole number written as a float (`2512.0`). It becomes `item_id BIGINT`, cast
  through `DOUBLE` because `CAST('2512.0' AS BIGINT)` fails under ANSI mode.
* **Empty attributes:** `adjective` (272 rows) and `modifier` (766 rows) are often empty, so they are nullable in layer 2.
* **Price:** many prices carry floating-point noise, such as `213.60000000000002`, so they become `DECIMAL(10,2)`.
  156 items have price 0.
* **Timestamps:** `created_at` always has the format `yyyy-MM-dd HH:mm:ss` and **contains no time-zone
  information**, so it becomes `TIMESTAMP_NTZ`.

## 2. Event source

In [4]:
# Row count, sample rows, and the number of rows per event_id
print(
    f"event rows: {event.count():,}, distinct event_id: {event.select('event_id').distinct().count():,}"
)
display(event.limit(5))
display(
    event.groupBy("event_id")
    .count()
    .groupBy(F.col("count").alias("rows_per_event"))
    .count()
)

event rows: 853,640, distinct event_id: 428,364


event_id,event_time,user_id,event.payload
ec5014f8a3aa402fbb207b22fac9121a,2014-06-16 15:23:05,110892.0,"{""event_name"":""test_assignment"",""platform"":""server"",""parameter_name"":""test_id"",""parameter_value"":""6""}"
ebf374a7668945db83cccc9e6adc86db,2014-05-17 15:23:05,272356.0,"{""event_name"":""test_assignment"",""platform"":""server"",""parameter_name"":""test_id"",""parameter_value"":""6""}"
bded8cc52707405c9dabe6452d20f8d6,2014-06-23 15:23:05,213337.0,"{""event_name"":""test_assignment"",""platform"":""server"",""parameter_name"":""test_id"",""parameter_value"":""6""}"
5bb6b791701b4ca8a435ed247fd14a00,2014-05-30 15:23:05,42912.0,"{""event_name"":""test_assignment"",""platform"":""server"",""parameter_name"":""test_id"",""parameter_value"":""6""}"
5c59cb9206f043b7bb9da75186f88b7e,2014-05-27 15:23:05,139790.0,"{""event_name"":""test_assignment"",""platform"":""server"",""parameter_name"":""test_id"",""parameter_value"":""6""}"


rows_per_event,count
2,425276
1,3088


In [5]:
# Parse the JSON payload: which keys exist, and which parameters and platforms each event type has
parsed = (
    event.withColumn(
        "payload_keys", F.expr("array_sort(json_object_keys(`event.payload`))")
    )
    .withColumn(
        "p",
        F.from_json(
            "`event.payload`",
            "event_name STRING, platform STRING, parameter_name STRING, parameter_value STRING",
        ),
    )
    .select("event_id", "event_time", "user_id", "payload_keys", "p.*")
)

display(parsed.groupBy("payload_keys").count())
display(
    parsed.groupBy("event_name", "parameter_name")
    .agg(
        F.count("*").alias("rows"),
        F.first("parameter_value").alias("example_value"),
        F.array_sort(F.collect_set("platform")).alias("platforms"),
    )
    .orderBy("event_name", "parameter_name")
)

payload_keys,count
"[""event_name"", ""parameter_name"", ""parameter_value"", ""platform""]",853640


event_name,parameter_name,rows,example_value,platforms
test_assignment,test_assignment,162490,1,"[""server""]"
test_assignment,test_id,162490,4,"[""server""]"
view_item,item_id,262786,3526,"[""android"", ""iOS"", ""mobile web"", ""web""]"
view_item,referrer,262786,home,"[""android"", ""iOS"", ""mobile web"", ""web""]"
view_user_profile,viewed_user_id,3088,70803,"[""android"", ""iOS"", ""mobile web"", ""web""]"


In [6]:
# Can the parameter rows be pivoted into one row per event without losing data?
per_event = parsed.groupBy("event_id").agg(
    F.count_distinct("event_time", "user_id", "event_name", "platform").alias(
        "distinct_event_attributes"
    ),
    F.count("*").alias("rows"),
    F.count_distinct("parameter_name").alias("distinct_parameters"),
    F.sort_array(F.collect_list("parameter_name")).alias("parameter_set"),
    F.first("event_name").alias("event_name"),
)
display(
    per_event.select(
        F.count_if(F.col("distinct_event_attributes") > 1).alias(
            "events_with_inconsistent_attributes"
        ),
        F.count_if(F.col("rows") != F.col("distinct_parameters")).alias(
            "events_with_repeated_parameter"
        ),
    )
)
display(per_event.groupBy("event_name", "parameter_set").count())

# ID-like payload parameters: are they plain digits, so they can be cast to BIGINT directly?
id_parameters = ["item_id", "test_id", "test_assignment", "viewed_user_id"]
display(
    parsed.filter(F.col("parameter_name").isin(id_parameters)).select(
        F.count_if(~F.col("parameter_value").rlike(r"^\d+$")).alias(
            "id_parameter_values_not_plain_digits"
        ),
    )
)

# Source formats of user_id and event_time
display(
    event.select(
        F.count_if(~F.col("user_id").rlike(r"^\d+\.0$")).alias("user_id_not_like_N.0"),
        F.count_if(
            ~F.col("event_time").rlike(r"^\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}$")
        ).alias("event_time_bad_format"),
        F.min("event_time").alias("min_event_time"),
        F.max("event_time").alias("max_event_time"),
    )
)

events_with_inconsistent_attributes,events_with_repeated_parameter
0,0


event_name,parameter_set,count
test_assignment,"[""test_assignment"", ""test_id""]",162490
view_item,"[""item_id"", ""referrer""]",262786
view_user_profile,"[""viewed_user_id""]",3088


id_parameter_values_not_plain_digits
0


user_id_not_like_N.0,event_time_bad_format,min_event_time,max_event_time
0,0,2013-03-09 09:28:22,2018-06-01 23:57:54


**Findings – event**

* **Grain:** there are 853,640 rows but only 428,364 distinct `event_id`. The file has **one row per event
  parameter**, so a logical event spans one or two rows.
* **Payload:** `event.payload` is JSON that always has the same four keys: `event_name`, `platform`,
  `parameter_name` and `parameter_value`. Every value is a string.
* **Event types:** each has a fixed set of parameters:

  | event_name | parameters | platforms |
  |---|---|---|
  | `view_item` | `item_id`, `referrer` | android, iOS, mobile web, web |
  | `test_assignment` | `test_id`, `test_assignment` | server |
  | `view_user_profile` | `viewed_user_id` | android, iOS, mobile web, web |

* **Safe to pivot:** within an event, time, user, event name and platform are identical, and no parameter repeats.
  Pivoting to one row per event (layer 2) therefore loses nothing.
* **ID formats:** IDs inside the payload are plain digits and are cast directly. `user_id` is written as a float
  (`178481.0`), so it is cast through `DOUBLE`.
* **Timestamps:** `event_time` has the format `yyyy-MM-dd HH:mm:ss` and **contains no time-zone information**
  (`TIMESTAMP_NTZ`). It runs from 2013-03-09 to 2018-06-01.

## 3. Relationship between item and event

In [7]:
# One row per item view, joined to the catalogue on the item id
views = parsed.filter("event_name = 'view_item' AND parameter_name = 'item_id'").select(
    "event_id",
    "event_time",
    "platform",
    F.col("parameter_value").cast("bigint").alias("item_id"),
)
items = item.select(
    F.col("id").cast("double").cast("bigint").alias("item_id"), "created_at"
)

print("view events:", views.count())
print(
    "viewed item_ids missing in item:",
    views.join(items, "item_id", "left_anti").count(),
)
print("items never viewed:", items.join(views, "item_id", "left_anti").count())
print(
    "views before item created_at:",
    views.join(items, "item_id").filter("event_time < created_at").count(),
)

view events: 262786
viewed item_ids missing in item: 0
items never viewed: 0
views before item created_at: 3148


**Findings – relationship**

* **Join key:** `item.id` matches the `item_id` parameter of `view_item` events. The two are written differently
  (`"2512.0"` vs `"2512"`), so both are cast to `BIGINT`.
* **Completeness:** every viewed item exists in `item`, and every item is viewed at least once.
* **Views before creation:** **3,148 views happen before the item's `created_at`.** The data can't tell which
  timestamp is wrong, so these views are kept and reported as a data-quality observation.

## 4. Checks relevant to the `top_item` datamart

In [8]:
views_by_year = views.withColumn("year", F.substring("event_time", 1, 4))

# Year coverage: are there partial years?
display(
    views_by_year.groupBy("year")
    .agg(
        F.count("*").alias("views"),
        F.min("event_time").alias("first_view"),
        F.max("event_time").alias("last_view"),
        F.count_distinct("item_id").alias("items_viewed"),
    )
    .orderBy("year")
)

# Rank ties: far fewer distinct yearly view counts than items means many items share a count
item_year = views_by_year.groupBy("item_id", "year").agg(F.count("*").alias("views"))
display(
    item_year.groupBy("year")
    .agg(
        F.count("*").alias("items"),
        F.count_distinct("views").alias("distinct_view_counts"),
        F.max("views").alias("max_views"),
    )
    .orderBy("year")
)

# Platform ties: item/years where several platforms share the highest view count
platform_views = views_by_year.groupBy("item_id", "year", "platform").agg(
    F.count("*").alias("views")
)
top_count = platform_views.groupBy("item_id", "year").agg(
    F.max("views").alias("max_views")
)
top_platforms = (
    platform_views.join(top_count, ["item_id", "year"])
    .filter("views = max_views")
    .groupBy("item_id", "year")
    .count()
)
display(
    top_platforms.select(
        F.count("*").alias("item_years"),
        F.count_if(F.col("count") > 1).alias("with_platform_tie"),
    )
)

year,views,first_view,last_view,items_viewed
2013,7448,2013-03-09 09:28:22,2013-12-31 22:45:31,2103
2014,32082,2014-01-01 00:55:05,2014-12-31 23:44:20,2198
2015,49462,2015-01-01 00:01:44,2015-12-31 23:58:08,2198
2016,63642,2016-01-01 00:02:08,2016-12-31 23:59:30,2198
2017,76314,2017-01-01 00:08:56,2017-12-31 23:48:39,2198
2018,33838,2018-01-01 00:04:14,2018-06-01 23:57:54,2198


year,items,distinct_view_counts,max_views
2013,2103,12,12
2014,2198,27,29
2015,2198,31,39
2016,2198,37,50
2017,2198,42,58
2018,2198,27,33


item_years,with_platform_tie
13093,1046


**Findings – datamart**

* **Partial years:** views cover 2013–2018. **2013 starts on 2013-03-09 and 2018 ends on 2018-06-01**, so both
  years are partial and are documented as such. 95 items have no views in 2013.
* **Rank ties:** many items share the same yearly view count (2015 has 2,198 items but only 31 distinct counts). Ties
  are therefore common, and the ranking has to define how they are handled (shared ranks).
* **Platform ties:** 1,046 of 13,093 item/years have **several platforms tied** for the most views, so the datamart
  lists all tied platforms.

## Summary: from findings to design

| Finding | Design in `merkle_de_case_study.ipynb` |
|---|---|
| One row per event parameter; attributes are consistent within an event | Layer 2 `event` pivoted to one row per event; a view is counted once per `event_id` |
| JSON payload with four fixed keys and five parameter names | Payload parsed with an explicit schema and pivoted into five typed columns |
| IDs written as `"2512.0"` (item, user) or plain digits (payload) | `BIGINT`, cast through `DOUBLE`; validated to have no fractional part; payload IDs cast directly |
| Price floating-point noise | `DECIMAL(10,2)` |
| Timestamp strings contain no time-zone information | `TIMESTAMP_NTZ` |
| Every viewed item exists in the catalogue | Simple join on `item_id`; validated in the main notebook |
| Views before item creation | Kept and reported as a data-quality observation |
| Partial years 2013 and 2018 | Included and documented in the datamart notes |
| Frequent rank ties | Shared ranks with `RANK()` |
| Platform ties in about 8% of item/years | `most_used_platforms` lists all tied platforms, sorted |

## 5. Post-run checks of the curated tables and the datamart

These queries read the layer 2 and layer 3 tables built by the main notebook. They are for review only and are not
part of the pipeline.

In [ ]:
item_curated = spark.table(f"{CATALOG}.layer2_curated.item_curated")
event_curated = spark.table(f"{CATALOG}.layer2_curated.event_curated")
top_item = spark.table(f"{CATALOG}.layer3_mart.top_item")

# Top 3 items per year, with item names added for readability
display(
    top_item.filter(F.col("view_rank") <= 3)
    .join(item_curated.select("item_id", "item_name"), "item_id")
    .select(
        "view_year",
        "view_rank",
        "item_id",
        "item_name",
        "total_views",
        "most_used_platforms",
    )
    .orderBy("view_year", "view_rank", "item_id")
)

In [ ]:
# Year coverage and views before the item's created_at, per year
display(
    event_curated.filter(F.col("event_name") == "view_item")
    .join(item_curated.select("item_id", "created_at"), "item_id")
    .groupBy("event_year")
    .agg(
        F.min("event_time").alias("first_view"),
        F.max("event_time").alias("last_view"),
        F.count("*").alias("views"),
        F.count_if(F.col("event_time") < F.col("created_at")).alias(
            "views_before_item_created"
        ),
    )
    .orderBy("event_year")
)